# 🛠️ Chapter 05-02b: Hands-On — Production RAG with Re-Ranking

---

## Level Up Your RAG Pipeline! 🚀

In this notebook, you'll implement advanced retrieval techniques.

By the end:
- ✅ Implemented multi-query retrieval
- ✅ Built hybrid search (vector + keyword)
- ✅ Added re-ranking for precision
- ✅ Compared basic vs advanced RAG quality

Let's build! 🚀

---

## 📦 Setup

In [ ]:
# !pip install langchain-openai langchain-community langchain-core chromadb python-dotenv rank_bm25

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ Ready!")

---

## 📄 Prepare a Knowledge Base

In [ ]:
# Simulate a company knowledge base
knowledge_base = [
    "DataPro is our flagship product launched in 2022. It provides real-time data analytics for businesses.",
    "DataPro pricing: Basic plan $99/month (up to 10 users), Pro plan $199/month (up to 50 users), Enterprise custom pricing.",
    "DataPro supports integrations with Salesforce, HubSpot, Slack, and 200+ apps via Zapier.",
    "Our API rate limit is 1000 requests per minute for Pro users and 5000 for Enterprise.",
    "For authentication, DataPro uses OAuth 2.0. API keys can be generated in Settings > Developer > API Keys.",
    "Common error: 'Rate limit exceeded' (HTTP 429). Solution: implement exponential backoff or upgrade plan.",
    "DataPro's uptime SLA is 99.9% for Pro and 99.99% for Enterprise plans.",
    "Data is encrypted at rest using AES-256 and in transit using TLS 1.3.",
    "To export data, go to Dashboard > Reports > Export. Supported formats: CSV, JSON, PDF.",
    "DataPro was named 'Best Analytics Tool 2024' by TechReview magazine.",
    "Our engineering team uses Python, React, PostgreSQL, and Redis in production.",
    "The machine learning features in DataPro use scikit-learn for predictions and anomaly detection.",
    "Customer support: email support@datapro.io, live chat on weekdays 9 AM - 6 PM EST.",
    "Bug reports should be filed at bugs.datapro.io with reproduction steps and screenshots.",
    "Q4 2024 revenue: $15M (up 40% YoY). Active users: 50,000 across 120 countries.",
]

# Create documents
docs = [Document(page_content=text, metadata={"source": f"kb_{i}", "index": i}) 
        for i, text in enumerate(knowledge_base)]

# Store in vector DB
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="advanced_rag")
print(f"✅ Loaded {len(docs)} knowledge base documents")

---

## 🧪 Technique 1: Multi-Query Retrieval

In [ ]:
# Generate multiple search queries from the user's question

multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", """Generate 3 different search queries to find information for the user's question.
Each query should approach the question from a different angle.
Return ONLY the queries, one per line. No numbering."""),
    ("human", "{question}")
])

def generate_queries(question: str) -> list:
    result = (multi_query_prompt | llm | StrOutputParser()).invoke({"question": question})
    queries = [q.strip() for q in result.strip().split("\n") if q.strip()]
    return queries

def multi_query_retrieve(question: str, k: int = 3) -> list:
    """Retrieve using multiple query variations."""
    queries = generate_queries(question)
    print(f"  📝 Generated queries:")
    for q in queries:
        print(f"     • {q}")
    
    # Retrieve for each query
    all_docs = []
    seen_content = set()
    for query in queries:
        results = vectorstore.similarity_search(query, k=k)
        for doc in results:
            if doc.page_content not in seen_content:
                all_docs.append(doc)
                seen_content.add(doc.page_content)
    
    return all_docs

# Test: compare basic vs multi-query
question = "How do I connect DataPro to my other tools?"

print(f"❓ {question}\n")
print("--- Basic Retrieval ---")
basic_results = vectorstore.similarity_search(question, k=3)
for doc in basic_results:
    print(f"  • {doc.page_content[:80]}...")

print("\n--- Multi-Query Retrieval ---")
multi_results = multi_query_retrieve(question)
for doc in multi_results:
    print(f"  • {doc.page_content[:80]}...")

---

## 🧪 Technique 2: BM25 + Vector Hybrid Search

In [ ]:
from rank_bm25 import BM25Okapi

# Prepare BM25 index
tokenized_docs = [doc.page_content.lower().split() for doc in docs]
bm25 = BM25Okapi(tokenized_docs)

def hybrid_search(query: str, k: int = 5, alpha: float = 0.5) -> list:
    """Combine vector search and BM25 keyword search.
    alpha: weight for vector scores (1-alpha for BM25)."""
    
    # Vector search
    vector_results = vectorstore.similarity_search_with_score(query, k=k)
    
    # BM25 search
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Normalize scores to 0-1
    if max(bm25_scores) > 0:
        bm25_scores = bm25_scores / max(bm25_scores)
    
    # Combine with Reciprocal Rank Fusion style
    combined_scores = {}
    for doc, score in vector_results:
        combined_scores[doc.page_content] = alpha * (1 - score)  # Convert distance to similarity
    
    for i, doc in enumerate(docs):
        bm25_score = (1 - alpha) * bm25_scores[i]
        if doc.page_content in combined_scores:
            combined_scores[doc.page_content] += bm25_score
        else:
            combined_scores[doc.page_content] = bm25_score
    
    # Sort and return top-k
    sorted_results = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    return [(content, score) for content, score in sorted_results[:k]]

# Test with a technical query that benefits from keyword matching
query = "HTTP 429 rate limit error"
print(f"🔍 Query: \"{query}\"\n")

print("--- Vector Search Only ---")
for doc, score in vectorstore.similarity_search_with_score(query, k=3):
    print(f"  [{1-score:.4f}] {doc.page_content[:80]}...")

print("\n--- Hybrid Search (Vector + BM25) ---")
for content, score in hybrid_search(query, k=3):
    print(f"  [{score:.4f}] {content[:80]}...")

---

## 🧪 Technique 3: LLM-Based Re-Ranking

In [ ]:
# Using the LLM itself as a re-ranker (works without external services)

rerank_prompt = ChatPromptTemplate.from_messages([
    ("system", """Rate how relevant this document is to the question on a scale of 0-10.
Return ONLY the number, nothing else."""),
    ("human", "Question: {question}\n\nDocument: {document}")
])

def rerank_documents(question: str, documents: list, top_k: int = 3) -> list:
    """Re-rank documents using LLM scoring."""
    scored_docs = []
    rerank_chain = rerank_prompt | llm | StrOutputParser()
    
    for doc_content in documents:
        content = doc_content if isinstance(doc_content, str) else doc_content.page_content
        try:
            score = float(rerank_chain.invoke({"question": question, "document": content}))
        except ValueError:
            score = 0
        scored_docs.append((content, score))
    
    # Sort by score descending
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return scored_docs[:top_k]

# Test: retrieve broadly, then re-rank precisely
question = "What security measures does DataPro implement?"
print(f"❓ {question}\n")

# Step 1: Broad retrieval (top 8)
initial_results = vectorstore.similarity_search(question, k=8)
print("--- Initial Retrieval (Top 8) ---")
for i, doc in enumerate(initial_results, 1):
    print(f"  {i}. {doc.page_content[:70]}...")

# Step 2: Re-rank to top 3
print("\n--- After Re-Ranking (Top 3) ---")
reranked = rerank_documents(question, initial_results)
for content, score in reranked:
    print(f"  [{score:.0f}/10] {content[:70]}...")

---

## 🚀 Build: Complete Advanced RAG Pipeline

In [ ]:
# Combine everything: Multi-Query + Hybrid + Re-rank

def advanced_rag(question: str) -> str:
    """Full advanced RAG pipeline."""
    print(f"\n{'='*60}")
    print(f"❓ {question}")
    print(f"{'='*60}")
    
    # Step 1: Multi-query retrieval
    print("\n📝 Step 1: Multi-Query Retrieval")
    all_docs = multi_query_retrieve(question, k=3)
    print(f"   → Retrieved {len(all_docs)} unique documents")
    
    # Step 2: Re-rank
    print("\n🏆 Step 2: Re-Ranking")
    top_docs = rerank_documents(question, all_docs, top_k=3)
    for content, score in top_docs:
        print(f"   [{score:.0f}/10] {content[:60]}...")
    
    # Step 3: Generate with context
    context = "\n\n".join(content for content, _ in top_docs)
    
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", """Answer based ONLY on the provided context. Be specific and cite information.
If the context doesn't answer the question, say so."""),
        ("human", "Context:\n{context}\n\nQuestion: {question}")
    ])
    
    chain = rag_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    print(f"\n💬 Answer: {answer}")
    return answer

# Test the full pipeline
advanced_rag("How do I handle rate limit errors in the API?")

---

## ✍️ Practice: Add HyDE to the Pipeline

**Challenge:** Implement HyDE (Hypothetical Document Embeddings) as an alternative retrieval strategy.

In [ ]:
# YOUR CODE HERE

# TODO: Create a prompt that generates a hypothetical answer
# TODO: Embed the hypothetical answer (not the question)
# TODO: Search the vector store with that embedding
# TODO: Compare results with basic retrieval

<details>
<summary>💡 Click to see solution</summary>

```python
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a short paragraph that would answer this question. Be specific."),
    ("human", "{question}")
])

def hyde_retrieve(question: str, k: int = 3):
    # Generate hypothetical document
    hypo_doc = (hyde_prompt | llm | StrOutputParser()).invoke({"question": question})
    print(f"Hypothetical doc: {hypo_doc[:100]}...")
    
    # Search with the hypothetical doc instead of question
    return vectorstore.similarity_search(hypo_doc, k=k)

results = hyde_retrieve("What integrations are supported?")
for doc in results:
    print(f"  • {doc.page_content[:80]}...")
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Multi-query → broader recall, catches different phrasings.     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Hybrid (vector + BM25) → exact term matches + semantic.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Re-ranking → retrieve broadly (50), re-rank precisely (5).     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 HyDE → embed a hypothetical answer, not the question.          │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Combine techniques for production-grade RAG.                   │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Production RAG — Evaluation & Debugging!

RAGAS framework, failure mode debugging, and the Enterprise RAG project.

---

### 🎉 Your RAG Is Production-Ready!
Multi-query, hybrid search, re-ranking — these are the techniques top companies use. 🚀